# Has America Gotten Better at Predicting Its Own Future?
### Forecast accuracy across eras of American history, 1946–2026 | BU RISE

**How to use:** Runtime → Run all. Everything downloads automatically.

Data: [Livingston Survey](https://www.philadelphiafed.org/surveys-and-data/real-time-data-research/livingston-survey) (Philadelphia Fed) — the oldest continuous survey of economic forecasts in the US, started 1946. [Documentation](https://www.philadelphiafed.org/-/media/FRBP/Assets/Surveys-And-Data/livingston-survey/livingston-documentation.pdf).

**File structure (verified):** `medians.xlsx` has one sheet per variable. Each sheet: `Date`, `VAR_BP` (actual value forecasters saw), `VAR_6M` / `VAR_12M` (forecasts), plus annual-horizon columns.

| Sheet | Variable | Coverage |
|---|---|---|
| CPI | Consumer Price Index | 1946– |
| IP | Industrial Production | 1946– |
| UNPR | Unemployment rate | 1961– |
| RGDPX | Real GDP | 1971– |
| GDPX | Nominal GDP/GNP | 1946– |
| TBILL, TBOND, PRIME | Interest rates | varies |

In [ ]:
# ============ 1. SETUP & DOWNLOAD ============
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests, io, warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True

BASE = 'https://www.philadelphiafed.org/-/media/FRBP/Assets/Surveys-And-Data/livingston-survey/historical-data/'
HEADERS = {'User-Agent': 'Mozilla/5.0 (research; BU RISE student project)'}

def download_xlsx(name):
    r = requests.get(BASE + name, headers=HEADERS, timeout=60)
    r.raise_for_status()
    print(f'{name}: {len(r.content):,} bytes')
    return pd.ExcelFile(io.BytesIO(r.content))

xl_med = download_xlsx('medians.xlsx')
print('Sheets (one per variable):', xl_med.sheet_names)

In [ ]:
# ============ 2. BUILD FORECAST-ERROR DATASET ============
ANALYZE   = ['CPI', 'UNPR', 'IP', 'RGDPX']          # levels + one rate; edit freely
RATE_VARS = {'UNPR', 'PRIME', 'TBOND', 'TBILL'}      # rates: use point differences, not % change

# Key idea: the actual outcome of a 12-month-ahead forecast is the base-period (_BP)
# value reported TWO SURVEYS LATER (surveys are semiannual). No second dataset needed.
rows = []
for v in ANALYZE:
    d = xl_med.parse(v).sort_values('Date').reset_index(drop=True)
    bp, f12 = d[f'{v}_BP'], d[f'{v}_12M']
    actual_next = bp.shift(-2)                        # realized value 12 months later
    if v in RATE_VARS:
        pred  = f12 - bp                              # predicted change, percentage points
        act   = actual_next - bp
        naive = bp - bp.shift(2)                      # naive: recent change repeats itself
    else:
        pred  = (f12 - bp) / bp * 100                 # predicted % growth
        act   = (actual_next - bp) / bp * 100
        naive = (bp - bp.shift(2)) / bp.shift(2) * 100
    rows.append(pd.DataFrame({
        'survey_date': pd.to_datetime(d['Date']), 'year': pd.to_datetime(d['Date']).dt.year,
        'variable': v, 'predicted': pred, 'actual': act, 'naive_forecast': naive}))

errors = pd.concat(rows, ignore_index=True).dropna(subset=['predicted', 'actual'])
errors['error'] = errors['predicted'] - errors['actual']        # signed -> bias
errors['abs_error'] = errors['error'].abs()
errors['naive_abs_error'] = (errors['naive_forecast'] - errors['actual']).abs()
print(errors.shape)
errors.groupby('variable')['survey_date'].agg(['min', 'max', 'count'])

### ⚠ Data-quality check: index rebasing

CPI and IP are *index numbers* whose base year was redefined a few times (e.g., CPI re-based in 1988). A rebase makes the level jump between two surveys, creating a **fake giant 'actual change'** that isn't real inflation. Inspect the biggest errors below — if they cluster at known rebase dates rather than real events, treat them as artifacts: either drop them (justify in methods!) or switch to the Fed's pre-computed `MedianGrowthRate.xlsx`, which handles base changes for you. This is a great 'limitations' talking point for the poster.

In [ ]:
# Top 12 largest errors: real shocks or rebasing artifacts?
sus = errors.nlargest(12, 'abs_error')[['survey_date', 'variable', 'predicted', 'actual', 'abs_error']]
print(sus.to_string(index=False))

# OPTIONAL cleaning rule — uncomment after inspecting, and report it in methods:
# ARTIFACT_DATES = []   # e.g. [('CPI', '1987-12-01'), ...]
# for var, dt in ARTIFACT_DATES:
#     errors = errors[~((errors.variable == var) & (errors.survey_date == dt))]

In [ ]:
# ============ 3. ERAS & SHOCKS ============
ERAS = {  # (start_inclusive, end_exclusive)
    'Postwar boom':          (1946, 1965),
    'Vietnam / stagflation': (1965, 1982),
    'Great Moderation':      (1982, 2000),
    'Terror / fin. crisis':  (2000, 2012),
    'Polarization / COVID':  (2012, 2027),
}
SHOCKS = {'Korea': 1950, 'Oil shock': 1973, 'Volcker/Oil II': 1979,
          'Gulf War': 1990, '9/11': 2001, 'Financial crisis': 2008, 'COVID': 2020}

errors['era'] = errors['year'].apply(
    lambda y: next((n for n, (a, b) in ERAS.items() if a <= y < b), None))
era_order = list(ERAS)
errors.groupby('era')['abs_error'].agg(['mean', 'median', 'count']).reindex(era_order).round(2)

In [ ]:
# ============ 4. HEADLINE FIGURE: rolling error across 80 years ============
fig, ax = plt.subplots(figsize=(14, 6))
for v in ANALYZE:
    s = errors[errors.variable == v].set_index('survey_date')['abs_error']
    ax.plot(s.rolling(10, min_periods=4).mean(), label=v, lw=2)
for name, (a, b) in ERAS.items():
    ax.axvline(pd.Timestamp(f'{a}-01-01'), color='gray', ls='--', alpha=.5)
    ax.text(pd.Timestamp(f'{a}-06-01'), ax.get_ylim()[1]*.95, name, fontsize=8, rotation=90, va='top')
for name, y in SHOCKS.items():
    ax.axvline(pd.Timestamp(f'{y}-06-01'), color='crimson', alpha=.25, lw=6)
ax.set_title('Absolute 12-month forecast error, rolling 10-survey mean (1946–2026)\nred bands = major world events', fontsize=13)
ax.set_ylabel('mean absolute error')
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ============ 5. ERA COMPARISON with bootstrap 95% CIs ============
def boot_ci(x, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    x = np.asarray(pd.Series(x).dropna())
    means = rng.choice(x, size=(n, len(x)), replace=True).mean(axis=1)
    return np.mean(x), *np.percentile(means, [2.5, 97.5])

fig, axes = plt.subplots(1, len(ANALYZE), figsize=(5*len(ANALYZE), 4.5))
axes = np.atleast_1d(axes)
for ax, v in zip(axes, ANALYZE):
    stats = []
    for era in era_order:
        grp = errors[(errors.era == era) & (errors.variable == v)]['abs_error']
        if len(grp) >= 5:
            m, lo, hi = boot_ci(grp)
            stats.append((era, m, lo, hi))
    names, means, los, his = zip(*stats)
    ax.bar(range(len(names)), means, color='steelblue', alpha=.8,
           yerr=[np.array(means)-np.array(los), np.array(his)-np.array(means)], capsize=4)
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=30, ha='right', fontsize=8)
    ax.set_title(v)
fig.suptitle('Mean absolute forecast error by era (bootstrap 95% CI)', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ============ 6. DID EXPERTISE MATTER? Economists vs. naive baseline ============
# Skill = 1 - (economist MAE / naive MAE). Positive = experts beat 'no-change' guessing.
skill = (errors.dropna(subset=['naive_abs_error'])
         .groupby(['era', 'variable'])
         .apply(lambda g: 1 - g['abs_error'].mean() / g['naive_abs_error'].mean())
         .unstack().reindex(era_order))
ax = skill.plot(kind='bar', figsize=(11, 4.5), rot=20)
ax.axhline(0, color='black', lw=1)
ax.set_title('Forecast skill vs. naive baseline by era  (>0 = economists beat guessing)')
ax.set_ylabel('skill score')
plt.tight_layout(); plt.show()
skill.round(2)

In [ ]:
# ============ 7. BIAS: which direction were they wrong? ============
bias = errors.groupby(['era', 'variable'])['error'].mean().unstack().reindex(era_order)
fig, ax = plt.subplots(figsize=(8, 4))
vmax = np.nanmax(np.abs(bias.values))
im = ax.imshow(bias.values, cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax)
ax.set_xticks(range(len(bias.columns))); ax.set_xticklabels(bias.columns)
ax.set_yticks(range(len(bias.index)));  ax.set_yticklabels(bias.index)
for i in range(bias.shape[0]):
    for j in range(bias.shape[1]):
        if not np.isnan(bias.values[i, j]):
            ax.text(j, i, f'{bias.values[i, j]:+.1f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, label='mean signed error (red = overpredicted)')
ax.set_title('Systematic bias by era: red = predicted too high, blue = too low')
plt.tight_layout(); plt.show()

In [ ]:
# ============ 8. ANATOMY OF A SURPRISE: shock recovery, aligned at t=0 ============
V = 'CPI'
e = errors[errors.variable == V].set_index('survey_date')['abs_error']
fig, axes = plt.subplots(2, 4, figsize=(15, 6), sharey=True)
for ax, (name, y) in zip(axes.flat, SHOCKS.items()):
    t0 = pd.Timestamp(f'{y}-06-01')
    w = e[(e.index >= t0 - pd.DateOffset(years=2)) & (e.index <= t0 + pd.DateOffset(years=4))]
    x = [(d - t0).days / 365.25 for d in w.index]
    ax.plot(x, w.values, 'o-', color='crimson')
    ax.axvline(0, color='black', ls='--', lw=1)
    ax.set_title(f'{name} ({y})', fontsize=10)
    ax.set_xlabel('years from event')
for ax in axes.flat[len(SHOCKS):]:
    ax.axis('off')
fig.suptitle(f'{V} forecast error around major world events — how long until forecasters recover?', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ============ 9. DO EXPERTS KNOW WHEN THEY DON'T KNOW? ============
# Disagreement (dispersion) at forecast time vs. the error of that same forecast.
try:
    xl_disp = download_xlsx('Dispersion2.xlsx')
    print('Dispersion sheets:', xl_disp.sheet_names)
    sheet = next(s for s in xl_disp.sheet_names if V in s)   # match sheet to variable
    dr = xl_disp.parse(sheet)
    print('Columns:', list(dr.columns))
    dcol = next(c for c in dr.columns if 'date' in str(c).lower())
    dv = next(c for c in dr.columns if '12' in str(c))       # 12-month-horizon dispersion
    dr['survey_date'] = pd.to_datetime(dr[dcol])
    merged = (errors[errors.variable == V]
              .merge(dr[['survey_date', dv]], on='survey_date')
              .dropna(subset=[dv, 'abs_error']))
    r = merged[dv].corr(merged['abs_error'])
    plt.figure(figsize=(7, 5))
    plt.scatter(merged[dv], merged['abs_error'], alpha=.5)
    plt.xlabel(f'economist disagreement at forecast time ({dv})')
    plt.ylabel('absolute error of that forecast')
    plt.title(f'Disagreement vs. error, {V}  (r = {r:.2f})\nPositive r = the expert community senses its own blind spots')
    plt.tight_layout(); plt.show()
except Exception as ex:
    print('Check sheet/column names printed above and adjust:', ex)

In [ ]:
# ============ 10. POLITICAL LAYER ============
# (a) Are election years harder to predict? (b) Do party transitions disrupt forecasts?
PARTY_TRANSITIONS = [1953, 1961, 1969, 1977, 1981, 1993, 2001, 2009, 2017, 2021, 2025]
errors['election_year'] = errors['year'] % 4 == 0
errors['transition'] = errors['year'].isin(PARTY_TRANSITIONS)

print('=== Mean absolute error ===')
print('Election years:    ', round(errors[errors.election_year]['abs_error'].mean(), 2))
print('Non-election years:', round(errors[~errors.election_year]['abs_error'].mean(), 2))
print('Transition years:  ', round(errors[errors.transition]['abs_error'].mean(), 2))
print('All other years:   ', round(errors[~errors.transition]['abs_error'].mean(), 2))

# Is the difference real? Bootstrap the gap.
rng = np.random.default_rng(1)
a = errors[errors.election_year]['abs_error'].dropna().values
b = errors[~errors.election_year]['abs_error'].dropna().values
gaps = [rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean() for _ in range(2000)]
lo, hi = np.percentile(gaps, [2.5, 97.5])
print(f'\nElection-year error gap: {a.mean()-b.mean():+.2f}  (95% CI [{lo:+.2f}, {hi:+.2f}])')
print('CI excludes zero -> difference is statistically meaningful' if lo*hi > 0 else 'CI includes zero -> no clear election effect')

# NOTE: raw comparison pools variables & eras — a confound! Better: compare within era,
# or within variable. Try: errors.groupby(['era','election_year'])['abs_error'].mean().unstack()

### Optional political upgrade: Economic Policy Uncertainty index

Download the US Historical News-Based EPU index (monthly, back to 1900) from [policyuncertainty.com](https://www.policyuncertainty.com/us_historical.html), upload it here, and correlate EPU at forecast time with subsequent forecast error. Directly tests: **does policy chaos make America unpredictable?**

## Interpretation checklist (write answers as you go)

1. Did mean absolute error decline across eras — or only between shocks?
2. In which eras did economists beat the naive baseline most? What was happening politically then?
3. Which biases match the era's politics (e.g., underestimating 1970s inflation)?
4. How many surveys does shock recovery take — and is it faster in recent decades?
5. Does disagreement predict error? What does that say about trusting confident experts?
6. Election years / party transitions: predictability cost, or no effect? (Watch the confound noted in cell 10.)

**Robustness (week 5):** shift era boundaries ±3 years and rerun. **Data caveat to report:** index rebasing artifacts (see the data-quality cell) and the changing survey panel over 80 years.